In [1]:
import json
from pathlib import Path

import imageio.v3 as iio
import matplotlib.pyplot as plt
import numpy as np
import pytorch3d.io
import pytorch3d.renderer
import pytorch3d.renderer.mesh
import pytorch3d.renderer.mesh.rasterizer
import pytorch3d.renderer.mesh.textures
import pytorch3d.structures
import torch

SORB_HDR = Path("/home/ahc/Datasets/Stanford-ORB/blender_HDR")
METHOD_DIR = Path("/home/ahc/Documents/metrology_ir/standard_NPBIR/stanford_orb")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

scenes = sorted([d.name for d in METHOD_DIR.iterdir() if (d / "pbir" / "mesh.obj").exists()])
print(scenes)

['blocks_scene006', 'cactus_scene007', 'car_scene004', 'gnome_scene003', 'grogu_scene002', 'pitcher_scene005', 'teapot_scene001']


In [ ]:
def render_normal(mesh_path, cameras, frame_idx, scale=4):
    mesh = pytorch3d.io.load_obj(mesh_path, device=device)
    vert = mesh[0]
    faces = mesh[1].verts_idx[None]
    mesh_torch = pytorch3d.structures.Meshes(verts=vert[None], faces=faces)
    mesh_torch.verts_normals_packed()
    normals = mesh_torch.verts_normals_padded()

    # scale intrinsics to full resolution
    K = torch.tensor(
        [
            [cameras["fx"] * scale, 0, cameras["cx"] * scale, 0],
            [0, cameras["fy"] * scale, cameras["cy"] * scale, 0],
            [0, 0, 1, 0],
            [0, 0, 0, 1],
        ],
        dtype=torch.float32,
        device=device,
    )
    cxcy = K[[0, 1], [2, 2]]
    image_size = [round(cxcy[1].item() * 2), round(cxcy[0].item() * 2)]

    frame = cameras["frames"][frame_idx]
    c2w = torch.tensor(frame["to_world"], dtype=torch.float32, device=device)
    c2w[:, :2] *= -1

    extrinsic = c2w.inverse()
    P = K @ extrinsic
    vert_ndc = torch.einsum("ij,nj->ni", P[:3, :3], vert) + P[:3, 3]
    vert_ndc[:, :2] = -(vert_ndc[:, :2] / vert_ndc[:, [2]] - cxcy) / cxcy.min()

    mesh_torch = pytorch3d.structures.Meshes(verts=vert_ndc[None], faces=faces)
    pix_to_face, zbuf, bary_coords, dists = pytorch3d.renderer.mesh.rasterize_meshes(
        mesh_torch,
        image_size=image_size,
        faces_per_pixel=1,
        perspective_correct=True,
        cull_backfaces=True,
    )
    frag = pytorch3d.renderer.mesh.rasterizer.Fragments(pix_to_face, zbuf, bary_coords, dists)
    mesh_torch.textures = pytorch3d.renderer.mesh.textures.TexturesVertex(
        torch.einsum("dnc,rc->dnr", normals, extrinsic[:3, :3])
    )
    rast_normal = mesh_torch.sample_textures(frag).squeeze().cpu().numpy()
    rast_normal[..., 1:] *= -1
    normal_img = np.clip((rast_normal + 1) * 0.5, 0, 1)
    mask = (pix_to_face.squeeze().cpu().numpy() >= 0)
    return normal_img, mask

In [ ]:
fig, axes = plt.subplots(len(scenes), 2, figsize=(12, 6 * len(scenes)))

for row, scene in enumerate(scenes):
    cam_path = SORB_HDR / scene / "cameras.json"
    mesh_path = METHOD_DIR / scene / "pbir" / "mesh.obj"

    with open(cam_path) as f:
        cameras = json.load(f)

    train_idx = cameras["split"]["train"][0]
    frame = cameras["frames"][train_idx]

    # derive full-res filename from 512x512 path (e.g. train_512x512/0001_image.exr -> train/0001.exr)
    stem = Path(frame["path"]).stem.replace("_image", "")
    gt_exr = iio.imread(SORB_HDR / scene / "train" / f"{stem}.exr")
    gt_mask = iio.imread(SORB_HDR / scene / "train_mask" / f"{stem}.png").astype(bool)
    gt_tonemapped = np.clip(gt_exr ** (1 / 2.2), 0, 1)
    gt_masked = gt_tonemapped * gt_mask[..., None]

    # render at 2048x2048 (scale=4 from 512x512 intrinsics)
    normal_img, render_mask = render_normal(mesh_path, cameras, train_idx, scale=4)
    normal_img[~render_mask] = 0

    axes[row, 0].imshow(gt_masked)
    axes[row, 0].set_title(f"{scene} — GT (masked, 2048x2048)")
    axes[row, 0].axis("off")

    axes[row, 1].imshow(normal_img)
    axes[row, 1].set_title(f"{scene} — PBIR geometry normal (2048x2048)")
    axes[row, 1].axis("off")

plt.tight_layout()
plt.savefig("viz_geometry.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
import imageio

OUT_DIR = Path("/home/ahc/Documents/metrology_ir/viz_masked")
OUT_DIR.mkdir(exist_ok=True)

for scene in scenes:
    cam_path = SORB_HDR / scene / "cameras.json"
    with open(cam_path) as f:
        cameras = json.load(f)

    train_idx = cameras["split"]["train"][0]
    frame = cameras["frames"][train_idx]
    stem = Path(frame["path"]).stem.replace("_image", "")

    gt_exr = iio.imread(SORB_HDR / scene / "train" / f"{stem}.exr")
    gt_mask = iio.imread(SORB_HDR / scene / "train_mask" / f"{stem}.png").astype(bool)
    gt_tonemapped = np.clip(gt_exr ** (1 / 2.2), 0, 1)
    gt_masked = (gt_tonemapped * gt_mask[..., None] * 255).astype(np.uint8)

    out_path = OUT_DIR / f"{scene}_masked.png"
    imageio.imwrite(str(out_path), gt_masked)
    print(f"saved {out_path}")


In [2]:
from irtk.scene import (
    EnvironmentLight, HDRFilm, Integrator, Mesh, MicrofacetBRDF, PerspectiveCameraFull, Scene,
)
from irtk.connectors.psdr_jit_connector import PSDRJITConnector

_psdr_connector = PSDRJITConnector()

def render_psdr(scene_name, cameras, frame_idx, scale=1, spp=64):
    """Render with psdr_jit using PBIR materials + reconstructed envmap."""
    pbir_base = METHOD_DIR / scene_name / "pbir"
    pbir_dir = next(
        (pbir_base / v / "final" for v in ["microfacet_basis-envmap_ls", "microfacet_naive-envmap_sg"]
         if (pbir_base / v / "final").exists()),
        None,
    )

    # image size: cameras.json intrinsics are for 512x512 train images
    base_w = base_h = 512
    w, h = base_w * scale, base_h * scale

    # normalized intrinsics are scale-independent (fx/w == fx*scale / (w*scale))
    fx_n = cameras["fx"] / base_w
    fy_n = cameras["fy"] / base_h
    cx_n = cameras["cx"] / base_w
    cy_n = cameras["cy"] / base_h

    to_world = np.array(cameras["frames"][frame_idx]["to_world"])

    irtk_scene = Scene()
    irtk_scene.set("sensor", PerspectiveCameraFull(fx_n, fy_n, cx_n, cy_n, to_world))

    diffuse   = iio.imread(str(pbir_dir / "diffuse.exr"))
    specular  = np.ones_like(diffuse) * 0.04
    roughness = iio.imread(str(pbir_dir / "roughness.exr"))[..., 0:1]  # single channel
    irtk_scene.set("mat", MicrofacetBRDF(diffuse, specular, roughness))

    irtk_scene.set("mesh", Mesh.from_file(str(pbir_dir / "mesh.obj"), "mat", use_face_normal=False))
    irtk_scene.set("envmap", EnvironmentLight.from_file(str(pbir_dir / "envmap.exr")))
    irtk_scene.set("film", HDRFilm(w, h))
    irtk_scene.set("integrator", Integrator("direct", {"mis": True}))

    render_opts = {
        "spp": spp, "sppe": 0, "sppse": 0,
        "log_level": 0, "npass": 1, "seed": 0,
        "guiding_options": {"type": "none"},
    }
    images = _psdr_connector.renderC(irtk_scene, render_opts, sensor_ids=[0])
    img = images[0].cpu().numpy()   # (H, W, 3), linear HDR
    mask = img.sum(-1) > 0
    return img, mask


In [3]:
OUT_PSDR_DIR = Path("/home/ahc/Documents/metrology_ir/viz_psdr_renders")
OUT_PSDR_DIR.mkdir(exist_ok=True)

for scene in scenes:
    cam_path = SORB_HDR / scene / "cameras.json"
    with open(cam_path) as f:
        cameras = json.load(f)

    train_idx = cameras["split"]["train"][0]
    img_hdr, mask = render_psdr(scene, cameras, train_idx, scale=2, spp=64)

    img_tm = np.clip(img_hdr ** (1 / 2.2), 0, 1)
    img_tm[~mask] = 0
    out = (img_tm * 255).astype(np.uint8)

    out_path = OUT_PSDR_DIR / f"{scene}_psdr.png"
    iio.imwrite(str(out_path), out)
    print(f"saved {out_path}")


saved /home/ahc/Documents/metrology_ir/viz_psdr_renders/blocks_scene006_psdr.png
saved /home/ahc/Documents/metrology_ir/viz_psdr_renders/cactus_scene007_psdr.png
saved /home/ahc/Documents/metrology_ir/viz_psdr_renders/car_scene004_psdr.png
saved /home/ahc/Documents/metrology_ir/viz_psdr_renders/gnome_scene003_psdr.png
saved /home/ahc/Documents/metrology_ir/viz_psdr_renders/grogu_scene002_psdr.png
saved /home/ahc/Documents/metrology_ir/viz_psdr_renders/pitcher_scene005_psdr.png
saved /home/ahc/Documents/metrology_ir/viz_psdr_renders/teapot_scene001_psdr.png
